# 🛰️ Aerial Surveillance — YOLOv8 Training Pipeline
**3-Stage Training: Transfer Learning → Fine-tune → Final**

Runtime → Change runtime type → **T4 GPU** before running!

In [ ]:
# CELL 1 — Check GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# CELL 2 — Upload project zip
from google.colab import files
uploaded = files.upload()  # upload aerial-colab.zip

In [ ]:
# CELL 3 — Unzip and enter folder
import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
# Find the project folder
for item in os.listdir('.'):
    if os.path.isdir(item) and 'aerial' in item.lower():
        os.chdir(item)
        break
print('Working dir:', os.getcwd())
print(os.listdir('.'))

In [ ]:
# CELL 4 — Install dependencies
!pip install ultralytics opencv-python Pillow flask numpy reportlab -q
print('✓ Dependencies installed')

In [ ]:
# CELL 5 — Upload VisDrone dataset
# Upload your VisDrone zip files here
from google.colab import files
print('Upload VisDrone2019-DET-train.zip')
uploaded_vd = files.upload()

In [ ]:
# CELL 6 — Extract VisDrone
import zipfile, os
for fname in uploaded_vd.keys():
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('visdrone_raw')
print('Extracted files:')
for item in os.listdir('visdrone_raw'):
    print(' ', item)

In [ ]:
# CELL 7 — Convert VisDrone to YOLO format
import sys
sys.path.insert(0, '.')
from src.data.visdrone_prep import prepare_visdrone
import os

# Find the extracted folders
raw = 'visdrone_raw'
folders = os.listdir(raw)
print('Available folders:', folders)

train_dir = None
val_dir   = None
test_dir  = None

for f in folders:
    fp = os.path.join(raw, f)
    if not os.path.isdir(fp): continue
    fl = f.lower()
    if 'train' in fl: train_dir = fp
    elif 'val'  in fl: val_dir   = fp
    elif 'test' in fl: test_dir  = fp

print(f'Train: {train_dir}')
print(f'Val:   {val_dir}')
print(f'Test:  {test_dir}')

result = prepare_visdrone(
    train_dir=train_dir,
    val_dir=val_dir,
    test_dir=test_dir,
    out_dir='dataset'
)
print('\nYAML written to:', result)

In [ ]:
# CELL 8 — STAGE 1: Transfer learning from COCO (100 epochs)
# This is the main training step — takes ~45 min on T4 GPU
!python src/model/train.py --stage 1 --epochs 100 --batch 16 --model-size s

In [ ]:
# CELL 9 — Check Stage 1 results
import json
try:
    with open('outputs/stage1_metrics.json') as f:
        m = json.load(f)
    print('Stage 1 Results:')
    print(f'  mAP@50:     {m["mAP50"]:.4f}')
    print(f'  Precision:  {m["precision"]:.4f}')
    print(f'  Recall:     {m["recall"]:.4f}')
except: print('Run Cell 8 first')

In [ ]:
# CELL 10 — STAGE 2: Fine-tune (50 epochs, frozen backbone)
# Takes ~20 min on T4 GPU
!python src/model/train.py --stage 2 --epochs 50 --batch 16

In [ ]:
# CELL 11 — STAGE 3: Final fine-tune (30 epochs, very low LR)
# Takes ~12 min on T4 GPU
!python src/model/train.py --stage 3 --epochs 30 --batch 16

In [ ]:
# CELL 12 — Final evaluation
!python src/model/evaluate.py --weights weights/aerial_yolov8.pt

In [ ]:
# CELL 13 — View training curves
import os
from IPython.display import Image, display
for stage in ['stage1', 'stage2', 'stage3']:
    results_png = f'outputs/{stage}/train/results.png'
    if os.path.exists(results_png):
        print(f'\n{stage.upper()} Training Curves:')
        display(Image(results_png))

In [ ]:
# CELL 14 — Download trained weights
from google.colab import files
import os
if os.path.exists('weights/aerial_yolov8.pt'):
    files.download('weights/aerial_yolov8.pt')
    print('Downloaded: aerial_yolov8.pt')
else:
    print('No weights found. Run training cells first.')